In [31]:
import download_data
import pandas as pd
import numpy as np

file_names = download_data.get_valeur_fonciere_path()

with open(file_names, "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline().strip())

pd.options.display.float_format = None

Utilisation des données en cache dans cache/valeur_fonciere
Identifiant de document|Reference document|1 Articles CGI|2 Articles CGI|3 Articles CGI|4 Articles CGI|5 Articles CGI|No disposition|Date mutation|Nature mutation|Valeur fonciere|No voie|B/T/Q|Type de voie|Code voie|Voie|Code postal|Commune|Code departement|Code commune|Prefixe de section|Section|No plan|No Volume|1er lot|Surface Carrez du 1er lot|2eme lot|Surface Carrez du 2eme lot|3eme lot|Surface Carrez du 3eme lot|4eme lot|Surface Carrez du 4eme lot|5eme lot|Surface Carrez du 5eme lot|Nombre de lots|Code type local|Type local|Identifiant local|Surface reelle bati|Nombre pieces principales|Nature culture|Nature culture speciale|Surface terrain
|||||||000001|02/01/2024|Vente|346,50||||B020|LE DELIVRE|1230|CHALEY|01|76||B|514||||||||||||0||||||P||99
|||||||000002|03/01/2024|Vente|10000,00||||B007|CHEVRY DESSOUS|1170|CHEVRY|01|103||B|1782||||||||||||0||||||S||115
|||||||000001|08/01/2024|Vente|249000,00||||B086|PIN HAMEAU|1290

In [32]:

df = pd.read_csv(file_names, sep="|")

/tmp/ipykernel_61884/4133842914.py:1: DtypeWarning: Columns (18,23,24,26,28,30,31,33,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_names, sep="|")


In [33]:
colonnes_utiles = [
    "Date mutation", 
    "Nature mutation", 
    "Valeur fonciere", 
    "No voie", 
    "Voie",
    "Type de voie",
    "Code voie",
    "Code postal", 
    "Commune",
    "B/T/Q",
    "Code commune",
    "Code departement",
    "Type local", 
    "Code type local",
    "Surface reelle bati", 
    "Nombre pieces principales", 
    "Surface terrain", 
    "1er lot",
    "2eme lot",
    "3eme lot",
    "4eme lot",
    "5eme lot"
]

df_filtre = df[colonnes_utiles]

print(df_filtre.head())


  Date mutation Nature mutation Valeur fonciere  No voie            Voie  \
0    02/01/2024           Vente          346,50      NaN      LE DELIVRE   
1    03/01/2024           Vente        10000,00      NaN  CHEVRY DESSOUS   
2    08/01/2024           Vente       249000,00      NaN      PIN HAMEAU   
3    03/01/2024           Vente       329500,00     29.0         DU JURA   
4    03/01/2024           Vente       329500,00   9001.0         DU JURA   

  Type de voie Code voie  Code postal Commune B/T/Q  ...   Type local  \
0          NaN      B020       1230.0  CHALEY   NaN  ...          NaN   
1          NaN      B007       1170.0  CHEVRY   NaN  ...          NaN   
2          NaN      B086       1290.0    LAIZ   NaN  ...          NaN   
3           PL      0500       1170.0     GEX   NaN  ...  Appartement   
4           PL      0500       1170.0     GEX   NaN  ...   Dépendance   

  Code type local Surface reelle bati  Nombre pieces principales  \
0             NaN                 Na

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3489149 entries, 0 to 3489148
Data columns (total 43 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Identifiant de document     float64
 1   Reference document          float64
 2   1 Articles CGI              float64
 3   2 Articles CGI              float64
 4   3 Articles CGI              float64
 5   4 Articles CGI              float64
 6   5 Articles CGI              float64
 7   No disposition              int64  
 8   Date mutation               object 
 9   Nature mutation             object 
 10  Valeur fonciere             object 
 11  No voie                     float64
 12  B/T/Q                       object 
 13  Type de voie                object 
 14  Code voie                   object 
 15  Voie                        object 
 16  Code postal                 float64
 17  Commune                     object 
 18  Code departement            object 
 19  Code commune         

In [35]:
#Vérification du pourcentage de valeurs manquantes dans Code departement avant de filtrer les data d idf
taux_na = df_filtre["Code departement"].isna().sum()
print(f"Nombre de valeurs manquantes dans Code departement : {taux_na}")

Nombre de valeurs manquantes dans Code departement : 0


In [36]:

#Nettoyage de la Colonne Code departement
df_filtre["Code departement"] = df_filtre["Code departement"].astype(str).str.strip()

# Filtrage des communes en ile de france 
codes_idf = ["75", "77", "78", "91", "92", "93", "94", "95"]
idf = df_filtre[df_filtre["Code departement"].isin(codes_idf)]

/tmp/ipykernel_61884/2204082965.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtre["Code departement"] = df_filtre["Code departement"].astype(str).str.strip()


In [37]:
valeurs_uniques = idf["Type local"].unique()
print(valeurs_uniques)

['Dépendance' 'Appartement' 'Maison' nan
 'Local industriel. commercial ou assimilé']


In [38]:
for col in idf.columns:
    taux_na = float(idf[col].isna().sum())
    print(f"Nombre de valeurs manquantes pour '{col}' : {taux_na}")


Nombre de valeurs manquantes pour 'Date mutation' : 0.0
Nombre de valeurs manquantes pour 'Nature mutation' : 0.0
Nombre de valeurs manquantes pour 'Valeur fonciere' : 2564.0
Nombre de valeurs manquantes pour 'No voie' : 55098.0
Nombre de valeurs manquantes pour 'Voie' : 6652.0
Nombre de valeurs manquantes pour 'Type de voie' : 44349.0
Nombre de valeurs manquantes pour 'Code voie' : 6652.0
Nombre de valeurs manquantes pour 'Code postal' : 6652.0
Nombre de valeurs manquantes pour 'Commune' : 0.0
Nombre de valeurs manquantes pour 'B/T/Q' : 393470.0
Nombre de valeurs manquantes pour 'Code commune' : 0.0
Nombre de valeurs manquantes pour 'Code departement' : 0.0
Nombre de valeurs manquantes pour 'Type local' : 82069.0
Nombre de valeurs manquantes pour 'Code type local' : 82069.0
Nombre de valeurs manquantes pour 'Surface reelle bati' : 82263.0
Nombre de valeurs manquantes pour 'Nombre pieces principales' : 82263.0
Nombre de valeurs manquantes pour 'Surface terrain' : 270235.0
Nombre de val

In [39]:
for col in idf.columns:
    taux_na = float(idf[col].isna().sum())
    print(f"Nombre de valeurs manquantes pour '{col}' : {taux_na}")

Nombre de valeurs manquantes pour 'Date mutation' : 0.0
Nombre de valeurs manquantes pour 'Nature mutation' : 0.0
Nombre de valeurs manquantes pour 'Valeur fonciere' : 2564.0
Nombre de valeurs manquantes pour 'No voie' : 55098.0
Nombre de valeurs manquantes pour 'Voie' : 6652.0
Nombre de valeurs manquantes pour 'Type de voie' : 44349.0
Nombre de valeurs manquantes pour 'Code voie' : 6652.0
Nombre de valeurs manquantes pour 'Code postal' : 6652.0
Nombre de valeurs manquantes pour 'Commune' : 0.0
Nombre de valeurs manquantes pour 'B/T/Q' : 393470.0
Nombre de valeurs manquantes pour 'Code commune' : 0.0
Nombre de valeurs manquantes pour 'Code departement' : 0.0
Nombre de valeurs manquantes pour 'Type local' : 82069.0
Nombre de valeurs manquantes pour 'Code type local' : 82069.0
Nombre de valeurs manquantes pour 'Surface reelle bati' : 82263.0
Nombre de valeurs manquantes pour 'Nombre pieces principales' : 82263.0
Nombre de valeurs manquantes pour 'Surface terrain' : 270235.0
Nombre de val

In [40]:
idf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 417261 entries, 2655797 to 3489148
Data columns (total 22 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Date mutation              417261 non-null  object 
 1   Nature mutation            417261 non-null  object 
 2   Valeur fonciere            414697 non-null  object 
 3   No voie                    362163 non-null  float64
 4   Voie                       410609 non-null  object 
 5   Type de voie               372912 non-null  object 
 6   Code voie                  410609 non-null  object 
 7   Code postal                410609 non-null  float64
 8   Commune                    417261 non-null  object 
 9   B/T/Q                      23791 non-null   object 
 10  Code commune               417261 non-null  int64  
 11  Code departement           417261 non-null  object 
 12  Type local                 335192 non-null  object 
 13  Code type local            

In [41]:
idf["x"] = pd.NA
idf["y"] = pd.NA

idf["Code postal"] = idf["Code postal"].astype(str).str.replace(".0", "", regex=False).str.strip()
idf["No voie"] = idf["No voie"].astype(str).str.replace(".0", "", regex=False).str.strip()

idf["Type de voie"] = idf["Type de voie"].fillna("").astype(str).str.strip()
idf["Voie"] = idf["Voie"].fillna("").astype(str).str.strip()

# Créer une colonne adresse propre
idf["adresse"] = (
    idf["No voie"] + " " +
    idf["Type de voie"] + " " +
    idf["Voie"]
).str.replace(" +", " ", regex=True).str.strip()

/tmp/ipykernel_61884/706913884.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idf["x"] = pd.NA
/tmp/ipykernel_61884/706913884.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idf["y"] = pd.NA
/tmp/ipykernel_61884/706913884.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v

In [42]:

idf = (
    idf
    .groupby(
        ["adresse", "Date mutation"], 
        as_index=False
    )
    .agg({
        "Valeur fonciere": "first", 
        "Code postal" : "first",   
        "Commune" : "first",          
        "Surface reelle bati": "sum",             
        "Surface terrain": "sum",                 
        "Nombre pieces principales": "sum",       
        "Code commune": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA ,                  
        "Type local": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA                     # info indicatif
    })
)

In [43]:
 idf = idf[idf["Surface reelle bati"].notna()]
 idf["Surface reelle bati"] = pd.to_numeric(idf["Surface reelle bati"])
 idf = idf[(idf["Surface reelle bati"] >= 9) & (idf["Surface reelle bati"] <= 300)]
 print(idf["Surface reelle bati"].min())
 idf = idf[idf["Type local"].isin(["Maison", "Appartement"])]

9.0


In [44]:

idf = idf.dropna(subset=["Valeur fonciere"])


idf["Valeur fonciere"] = (
    idf["Valeur fonciere"]
        .astype(str)
        .str.replace(" ", "", regex=False)    
        .str.replace(",", ".", regex=False)    
        .str.replace("\xa0", "", regex=False)  
        .str.extract(r'(\d+\.?\d*)')[0]        
        .astype(float)
)
idf["Valeur foncière au mètre carré"] = (
    idf["Valeur fonciere"] / idf["Surface reelle bati"]
)

deciles_avant = idf["Valeur foncière au mètre carré"].quantile([0.1 * i for i in range(1, 11)])
print("Déciles avant filtrage :")
deciles_avant = deciles_avant.apply(lambda x: f"{x:,.2f} €")
print(deciles_avant.to_string())

n_before = len(idf)
prix_min_avant = idf["Valeur foncière au mètre carré"].min()
prix_max_avant = idf["Valeur foncière au mètre carré"].max()
print(f"Nombre de lignes avant filtrage : {n_before}")
print(f"Valeur foncière au mètre carré minimale avant filtrage : {prix_min_avant}")
print(f"Valeur foncière au mètre carré maximal avant filtrage : {prix_max_avant}")

idf = idf.loc[
    (idf["Valeur foncière au mètre carré"] >= 1000) &
    (idf["Valeur foncière au mètre carré"] <= 20000)
]

deciles_apres = idf["Valeur foncière au mètre carré"].quantile([0.1 * i for i in range(1, 11)])
print("\nDéciles après filtrage :")
deciles_apres = deciles_apres.apply(lambda x: f"{x:,.2f} €")
print(deciles_apres.to_string())


n_after = len(idf)
n_supprimees = n_before - n_after
print(f"Nombre de lignes supprimées : {n_supprimees}")
prix_min_apres = idf["Valeur foncière au mètre carré"].min()
prix_max_apres = idf["Valeur foncière au mètre carré"].max()
print(f"Valeur foncière au mètre carré minimale après filtrage : {prix_min_apres}")
print(f"Valeur foncière au mètre carré maximal avant filtrage : {prix_max_apres}")

Déciles avant filtrage :
0.1        2,064.48 €
0.2        2,755.48 €
0.3        3,261.52 €
0.4        3,769.23 €
0.5        4,405.70 €
0.6        5,324.22 €
0.7        6,674.42 €
0.8        8,333.33 €
0.9       10,348.79 €
1.0    2,247,428.57 €
Nombre de lignes avant filtrage : 75944
Valeur foncière au mètre carré minimale avant filtrage : 0.0012096774193548388
Valeur foncière au mètre carré maximal avant filtrage : 2247428.5714285714

Déciles après filtrage :
0.1     2,238.92 €
0.2     2,846.25 €
0.3     3,333.33 €
0.4     3,833.33 €
0.5     4,468.09 €
0.6     5,380.95 €
0.7     6,720.43 €
0.8     8,323.53 €
0.9    10,260.00 €
1.0    20,000.00 €
Nombre de lignes supprimées : 2243
Valeur foncière au mètre carré minimale après filtrage : 1000.0
Valeur foncière au mètre carré maximal avant filtrage : 20000.0


In [45]:
percentiles = np.arange(1, 100, 1)  # 1%, 2%, ..., 99%
valeurs_percentiles = idf["Valeur foncière au mètre carré"].quantile(percentiles / 100.0)
for p, v in zip(percentiles, valeurs_percentiles):
    print(f"Percentile {p}% : {v:,.2f} €/m²")

Percentile 1% : 1,219.51 €/m²
Percentile 2% : 1,384.62 €/m²
Percentile 3% : 1,538.46 €/m²
Percentile 4% : 1,666.67 €/m²
Percentile 5% : 1,785.71 €/m²
Percentile 6% : 1,885.96 €/m²
Percentile 7% : 1,980.20 €/m²
Percentile 8% : 2,071.26 €/m²
Percentile 9% : 2,158.54 €/m²
Percentile 10% : 2,238.92 €/m²
Percentile 11% : 2,312.50 €/m²
Percentile 12% : 2,380.95 €/m²
Percentile 13% : 2,446.99 €/m²
Percentile 14% : 2,508.74 €/m²
Percentile 15% : 2,571.43 €/m²
Percentile 16% : 2,631.58 €/m²
Percentile 17% : 2,687.50 €/m²
Percentile 18% : 2,743.88 €/m²
Percentile 19% : 2,794.44 €/m²
Percentile 20% : 2,846.25 €/m²
Percentile 21% : 2,894.74 €/m²
Percentile 22% : 2,944.44 €/m²
Percentile 23% : 3,000.00 €/m²
Percentile 24% : 3,043.01 €/m²
Percentile 25% : 3,090.91 €/m²
Percentile 26% : 3,137.25 €/m²
Percentile 27% : 3,187.50 €/m²
Percentile 28% : 3,235.29 €/m²
Percentile 29% : 3,281.67 €/m²
Percentile 30% : 3,333.33 €/m²
Percentile 31% : 3,378.38 €/m²
Percentile 32% : 3,422.22 €/m²
Percentile 33% : 

In [46]:
idf["Commune"] = idf["Commune"].str.replace(r"PARIS \d{1,2}", "PARIS", regex=True)

idf = idf.rename(columns={
    "Code postal": "Code_postal",
    "adresse": "adresse",   # juste pour s'assurer que c'est correct
    "Commune": "Commune"
})

idf["search"] = idf["adresse"].astype(str) + " " + idf["Code_postal"].astype(str) + " " + idf["Commune"]
idfs = idf.iloc[55000: 55010].copy()
idf_search = idfs["search"]
idf_search.to_csv("idfs.csv", index=False, encoding="utf-8")

In [48]:
import pandas as pd
import requests


# Payload pour l'API
payload = {
    "indexes": ["address"],
}

# Envoyer le CSV à l'API
url = "https://api-adresse.data.gouv.fr/search/csv/"
files = {"data": open("idfs.csv", "rb")}
response = requests.post(url, files=files)

if response.status_code == 200:
    # Sauvegarde le CSV retourné tel quel
    with open("idfs_geocoded.csv", "wb") as f:
        f.write(response.content)
    print("CSV géocodé sauvegardé avec succès !")
else:
    print("Erreur API :", response.text)

idfs_geocoded = pd.read_csv("idfs_geocoded.csv")


idfs_geocoded



CSV géocodé sauvegardé avec succès !


,search,longitude,latitude,result_score,result_score_next,result_label,result_type,result_id,result_housenumber,result_name,result_street,result_postcode,result_city,result_context,result_citycode,result_oldcitycode,result_oldcity,result_district,result_status
0,53 RUE DES GRANDES VIGNES 91310 MONTLHERY,2.256793,48.639723,0.963200,0.604830,53 Rue des Grandes Vignes 91310 Montlhéry,housenumber,91425_0001_00053,53,53 Rue des Grandes Vignes,Rue des Grandes Vignes,91310,Montlhéry,"91, Essonne, Île-de-France",91425,NaN,NaN,NaN,ok
1,53 RUE DES ILES 77176 SAVIGNY-LE-TEMPLE,2.564219,48.598785,0.969811,0.753404,53 Rue des Iles 77176 Savigny-le-Temple,housenumber,77445_1380_00053,53,53 Rue des Iles,Rue des Iles,77176,Savigny-le-Temple,"77, Seine-et-Marne, Île-de-France",77445,NaN,NaN,NaN,ok
2,53 RUE DES PRAIRIES 75020 PARIS,2.402185,48.863218,0.980177,NaN,53 Rue des Prairies 75020 Paris,housenumber,75120_7785_00053,53,53 Rue des Prairies,Rue des Prairies,75020,Paris,"75, Paris, Île-de-France",75120,NaN,NaN,Paris 20e Arrondissement,ok
3,53 RUE DES RUELLES 91300 MASSY,2.256426,48.729554,0.971379,0.631075,53 Rue des Ruelles 91300 Massy,housenumber,91377_4210_00053,53,53 Rue des Ruelles,Rue des Ruelles,91300,Massy,"91, Essonne, Île-de-France",91377,NaN,NaN,NaN,ok
4,53 RUE DES RUISSEAUX 95370 MONTIGNY LES CORMEI...,2.191261,48.987372,0.965380,NaN,53 Rue des Ruisseaux 95370 Montigny-lès-Cormei...,housenumber,95424_0727_00053,53,53 Rue des Ruisseaux,Rue des Ruisseaux,95370,Montigny-lès-Cormeilles,"95, Val-d'Oise, Île-de-France",95424,NaN,NaN,NaN,ok
5,53 RUE DES VOIES DU BOIS 92700 COLOMBES,2.249572,48.919020,0.980203,NaN,53 Rue des Voies du Bois 92700 Colombes,housenumber,92025_9680_00053,53,53 Rue des Voies du Bois,Rue des Voies du Bois,92700,Colombes,"92, Hauts-de-Seine, Île-de-France",92025,NaN,NaN,NaN,ok
6,53 RUE DESGRANGES 93100 MONTREUIL,2.443186,48.851972,0.974196,NaN,53 Rue Desgranges 93100 Montreuil,housenumber,93048_2540_00053,53,53 Rue Desgranges,Rue Desgranges,93100,Montreuil,"93, Seine-Saint-Denis, Île-de-France",93048,NaN,NaN,NaN,ok
7,53 RUE DESIRE CLEMENT 78700 CONFLANS SAINTE HO...,2.110208,49.001661,0.974154,0.720815,53 Rue Désiré Clément 78700 Conflans-Sainte-Ho...,housenumber,78172_0790_00053,53,53 Rue Désiré Clément,Rue Désiré Clément,78700,Conflans-Sainte-Honorine,"78, Yvelines, Île-de-France",78172,NaN,NaN,NaN,ok
8,53 RUE DIDOT 75014 PARIS,2.319799,48.831124,0.980175,0.499823,53 Rue Didot 75014 Paris,housenumber,75114_2794_00053,53,53 Rue Didot,Rue Didot,75014,Paris,"75, Paris, Île-de-France",75114,NaN,NaN,Paris 14e Arrondissement,ok
9,53 RUE DOUDEAUVILLE 75018 PARIS,2.352556,48.888311,0.980268,NaN,53 Rue Doudeauville 75018 Paris,housenumber,75118_2921_00053,53,53 Rue Doudeauville,Rue Doudeauville,75018,Paris,"75, Paris, Île-de-France",75118,NaN,NaN,Paris 18e Arrondissement,ok


In [49]:

idfs = idfs.merge(
    idfs_geocoded,
    on=["search"], 
    how="left"
)

In [50]:
idfs

,adresse,Date mutation,Valeur fonciere,Code_postal,Commune,Surface reelle bati,Surface terrain,Nombre pieces principales,Code commune,Type local,...,result_name,result_street,result_postcode,result_city,result_context,result_citycode,result_oldcitycode,result_oldcity,result_district,result_status
0,53 RUE DES GRANDES VIGNES,19/06/2024,462500.0,91310,MONTLHERY,142.0,559.0,5.0,425,Maison,...,53 Rue des Grandes Vignes,Rue des Grandes Vignes,91310,Montlhéry,"91, Essonne, Île-de-France",91425,NaN,NaN,NaN,ok
1,53 RUE DES ILES,19/12/2024,260000.0,77176,SAVIGNY-LE-TEMPLE,83.0,315.0,5.0,445,Maison,...,53 Rue des Iles,Rue des Iles,77176,Savigny-le-Temple,"77, Seine-et-Marne, Île-de-France",77445,NaN,NaN,NaN,ok
2,53 RUE DES PRAIRIES,27/09/2024,240000.0,75020,PARIS,27.0,0.0,1.0,120,Appartement,...,53 Rue des Prairies,Rue des Prairies,75020,Paris,"75, Paris, Île-de-France",75120,NaN,NaN,Paris 20e Arrondissement,ok
3,53 RUE DES RUELLES,05/11/2024,336000.0,91300,MASSY,95.0,891.0,5.0,377,Maison,...,53 Rue des Ruelles,Rue des Ruelles,91300,Massy,"91, Essonne, Île-de-France",91377,NaN,NaN,NaN,ok
4,53 RUE DES RUISSEAUX,15/03/2024,171800.0,95370,MONTIGNY LES CORMEILLES,46.0,0.0,2.0,424,Appartement,...,53 Rue des Ruisseaux,Rue des Ruisseaux,95370,Montigny-lès-Cormeilles,"95, Val-d'Oise, Île-de-France",95424,NaN,NaN,NaN,ok
5,53 RUE DES VOIES DU BOIS,07/05/2024,254000.0,92700,COLOMBES,38.0,47.0,2.0,25,Maison,...,53 Rue des Voies du Bois,Rue des Voies du Bois,92700,Colombes,"92, Hauts-de-Seine, Île-de-France",92025,NaN,NaN,NaN,ok
6,53 RUE DESGRANGES,29/03/2024,747410.0,93100,MONTREUIL,88.0,45.0,6.0,48,Maison,...,53 Rue Desgranges,Rue Desgranges,93100,Montreuil,"93, Seine-Saint-Denis, Île-de-France",93048,NaN,NaN,NaN,ok
7,53 RUE DESIRE CLEMENT,31/01/2024,225000.0,78700,CONFLANS SAINTE HONORINE,66.0,0.0,3.0,172,Appartement,...,53 Rue Désiré Clément,Rue Désiré Clément,78700,Conflans-Sainte-Honorine,"78, Yvelines, Île-de-France",78172,NaN,NaN,NaN,ok
8,53 RUE DIDOT,21/10/2024,593000.0,75014,PARIS,55.0,0.0,3.0,114,Appartement,...,53 Rue Didot,Rue Didot,75014,Paris,"75, Paris, Île-de-France",75114,NaN,NaN,Paris 14e Arrondissement,ok
9,53 RUE DOUDEAUVILLE,02/08/2024,379225.0,75018,PARIS,43.0,0.0,3.0,118,Appartement,...,53 Rue Doudeauville,Rue Doudeauville,75018,Paris,"75, Paris, Île-de-France",75118,NaN,NaN,Paris 18e Arrondissement,ok
